# 🚀 AI Speaking Tutor 3D — Pipeline Fine-Tune Trên Colab A100 (v2, chạy qua đêm)

**Cách dùng:** `Runtime → Change runtime type → A100 GPU + RAM cao cấp → Save` rồi `Runtime → Run all`.
Khi popup Google Drive hiện ra, bấm **Connect / Allow**. Sau đó có thể đi ngủ.

| Giai đoạn | Nội dung | Dữ liệu | Thời gian ước tính (A100 80GB) |
| :--- | :--- | :--- | :--- |
| 0 | Mount Drive, kiểm tra GPU, tải dữ liệu | SpeechOcean762 (HuggingFace, tự tải) | ~5–10 phút |
| 1 | Wav2Vec2-XLSR-53 chấm điểm câu: accuracy, fluency, completeness, prosodic, total | SpeechOcean762 | ~25–40 phút |
| 2 | Wav2Vec2-XLSR-53 nhận dạng âm vị thực tế (CTC) → phát hiện thay thế / nuốt âm | SpeechOcean762 (+ L2-ARCTIC tiếng Việt nếu có) | ~40–60 phút |
| 3 | Llama 3.1 8B QLoRA (Unsloth) gia sư tiếng Việt + `visemeTimeline` 3D, xuất GGUF | Dataset sư phạm tự sinh (~2.400 mẫu) | ~40–70 phút |

**Cơ chế an toàn:**
- Mỗi giai đoạn bọc `try/except`: một giai đoạn lỗi **không** làm dừng các giai đoạn sau.
- Kết quả từng giai đoạn được chép ngay vào `MyDrive/AI_Speaking_Tutor_Models/` và đánh dấu `_DONE.json`.
- Nếu Colab bị ngắt, chỉ cần `Run all` lại: giai đoạn đã xong sẽ **tự bỏ qua**.
- Nhật ký đầy đủ: `MyDrive/AI_Speaking_Tutor_Models/training_log.txt`; trạng thái: `run_status.json`.
- Khi xong toàn bộ, runtime **tự ngắt** để không tốn thêm đơn vị điện toán (tắt bằng `AUTO_DISCONNECT_WHEN_DONE = False`).

**L2-ARCTIC (Tự động tải 100% từ Hugging Face):** Pipeline tự động tải trực tiếp 4 bộ giọng phát âm người Việt (`HQTV, PNV, THV, TLV`) từ Hugging Face mirror (`chikingsley/l2-arctic-release-v5.0`) hoàn toàn miễn phí, **KHÔNG CẦN ĐIỀN FORM, KHÔNG CẦN TÀI KHOẢN TRƯỜNG HỌC, KHÔNG CẦN TẢI THỦ CÔNG**.

## Ô 1 — Cấu hình

In [ ]:
# ===================== CẤU HÌNH (có thể giữ nguyên) =====================
DRIVE_ROOT = "/content/drive/MyDrive/AI_Speaking_Tutor_Models"
L2ARCTIC_DRIVE_DIR = "/content/drive/MyDrive/AI_Speaking_Tutor_Datasets/l2arctic"  # tùy chọn
LOCAL_ROOT = "/content/ast_work"

RUN_PHASE1 = True
RUN_PHASE2 = True
RUN_PHASE3 = True
EXPORT_GGUF = True                 # xuất Llama sang GGUF q4_k_m để chạy local (Ollama / llama.cpp)
AUTO_DISCONNECT_WHEN_DONE = True   # tự ngắt runtime khi xong để tiết kiệm đơn vị điện toán

SEED = 42
BASE_W2V = "facebook/wav2vec2-large-xlsr-53"
MAX_AUDIO_SEC = 15.0

P1_EPOCHS = 15
P1_BATCH = 8
P1_LR_BACKBONE = 2e-5
P1_LR_HEAD = 1e-3

P2_EPOCHS = 25
P2_BATCH = 8
P2_LR = 1e-4
P2_EVAL_EVERY = 5

P3_MAX_STEPS = 500
P3_NUM_SAMPLES = 2400
print("Đã nạp cấu hình.")

## Ô 2 — Mount Google Drive, kiểm tra GPU, hàm tiện ích

In [ ]:
import os, sys, json, time, shutil, traceback, subprocess, gc, random, math, io, re, glob, zipfile
import importlib.util
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)
LOG_FILE = os.path.join(DRIVE_ROOT, "training_log.txt")
STATUS_FILE = os.path.join(DRIVE_ROOT, "run_status.json")


def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception as e:
        print(f"(không ghi được log vào Drive: {e})")


def load_status():
    if os.path.exists(STATUS_FILE):
        try:
            with open(STATUS_FILE, encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}


def set_status(key, value):
    s = load_status()
    s[key] = value
    s["updated_at"] = datetime.now().isoformat()
    with open(STATUS_FILE, "w", encoding="utf-8") as f:
        json.dump(s, f, indent=2, ensure_ascii=False)


def phase_done(name):
    return os.path.exists(os.path.join(DRIVE_ROOT, name, "_DONE.json"))


def mark_done(name, info):
    os.makedirs(os.path.join(DRIVE_ROOT, name), exist_ok=True)
    with open(os.path.join(DRIVE_ROOT, name, "_DONE.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)


def sync_to_drive(local_dir, name):
    dst = os.path.join(DRIVE_ROOT, name)
    shutil.copytree(local_dir, dst, dirs_exist_ok=True)
    log(f"Đã chép {local_dir} -> {dst}")
    return dst


def free_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def pip_install(packages, timeout_sec):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(packages)
    log("pip install " + " ".join(packages))
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
    if r.returncode != 0:
        log(r.stdout[-2000:])
        log(r.stderr[-4000:])
        raise RuntimeError(f"pip install thất bại: {packages}")


import torch
assert torch.cuda.is_available(), "Chưa bật GPU! Runtime -> Change runtime type -> A100 GPU + RAM cao cấp"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
log("=" * 70)
log(f"BẮT ĐẦU PHIÊN MỚI | GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB:.1f} GB | torch {torch.__version__}")
set_status("session_started", datetime.now().isoformat())

## Ô 3 — Kiểm tra thư viện cho Giai đoạn 1–2 (dùng sẵn của Colab, không nâng cấp torch)

In [ ]:
try:
    need = [("datasets", "datasets"), ("soundfile", "soundfile"), ("librosa", "librosa"), ("transformers", "transformers")]
    missing = [pkg for pkg, mod in need if importlib.util.find_spec(mod) is None]
    if missing:
        pip_install(missing, 900)
    import datasets, soundfile, librosa, transformers
    log(f"transformers {transformers.__version__} | datasets {datasets.__version__} | librosa {librosa.__version__}")
    set_status("install_phase12", "ok")
except Exception:
    log(traceback.format_exc())
    set_status("install_phase12", "failed")

## Ô 4 — Tải & tiền xử lý dữ liệu (SpeechOcean762 + L2-ARCTIC tùy chọn)

In [ ]:
import numpy as np

SCORE_FIELDS = ["accuracy", "fluency", "completeness", "prosodic", "total"]
VN_SPEAKERS = ["HQTV", "PNV", "THV", "TLV"]
DATA_OK = False
SO_TRAIN, SO_TEST, L2_ITEMS = [], [], []
SCORE_MAX = [10.0] * len(SCORE_FIELDS)


def to_16k_mono(wav, sr):
    import librosa
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=16000)
    return wav[: int(MAX_AUDIO_SEC * 16000)].astype(np.float32)


def decode_audio_field(audio_field):
    import soundfile as sf
    raw = audio_field.get("bytes")
    if raw is not None:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    else:
        wav, sr = sf.read(audio_field["path"], dtype="float32", always_2d=False)
    return to_16k_mono(wav, sr)


def strip_stress(p):
    return re.sub(r"\d", "", str(p).strip().upper())


def actual_phones_from_words(words):
    seq = []
    for w in words or []:
        phones = [strip_stress(p) for p in (w.get("phones") or [])]
        for m in w.get("mispronunciations") or []:
            idx = m.get("index")
            pp = str(m.get("pronounced-phone", "")).strip()
            if not isinstance(idx, int) or not (0 <= idx < len(phones)):
                continue
            if "DEL" in pp.upper():
                phones[idx] = None
            elif pp == "" or pp.startswith("<"):
                phones[idx] = "SPN"
            else:
                phones[idx] = strip_stress(pp)
        seq.extend([p for p in phones if p])
    return seq


def prepare_split(split, name):
    items, errors = [], 0
    for i, ex in enumerate(split):
        try:
            wav = decode_audio_field(ex["audio"])
            if len(wav) < 8000:
                continue
            scores = [float(ex.get(k) or 0.0) for k in SCORE_FIELDS]
            items.append({"wav": wav, "scores": scores, "text": ex.get("text", ""),
                          "seq": actual_phones_from_words(ex.get("words"))})
        except Exception as e:
            errors += 1
            if errors <= 5:
                log(f"Bỏ qua mẫu {name}[{i}]: {e}")
    log(f"SpeechOcean762 {name}: {len(items)} mẫu hợp lệ, {errors} lỗi")
    return items


def read_textgrid_text(path):
    with open(path, "rb") as f:
        raw = f.read()
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return raw.decode("utf-16", errors="ignore")
    return raw.decode("utf-8", errors="ignore")


def parse_l2arctic_phones(path):
    txt = read_textgrid_text(path)
    m = re.search(r'name\s*=\s*"phones"(.*?)(?=item\s*\[\d+\]|\Z)', txt, re.S | re.I)
    if not m:
        return None
    seq = []
    for lab in re.findall(r'text\s*=\s*"(.*?)"', m.group(1)):
        lab = lab.strip()
        if not lab:
            continue
        parts = [p.strip() for p in lab.split(",")]
        if len(parts) >= 3:
            typ = parts[2].lower()
            if typ.startswith("d"):
                continue
            ph = parts[1].replace("*", "")
            if ph.lower() == "err":
                seq.append("SPN")
                continue
        else:
            ph = parts[0].replace("*", "")
        ph = strip_stress(ph)
        if ph in ("SIL", "SP", "SPN", ""):
            continue
        seq.append(ph)
    return seq


def extract_l2arctic_archives():
    target = os.path.join(LOCAL_ROOT, "l2arctic")
    os.makedirs(target, exist_ok=True)
    roots = []
    if os.path.isdir(L2ARCTIC_DRIVE_DIR):
        roots.append(L2ARCTIC_DRIVE_DIR)
        for z in glob.glob(os.path.join(L2ARCTIC_DRIVE_DIR, "**", "*.zip"), recursive=True):
            try:
                with zipfile.ZipFile(z) as zf:
                    members = [n for n in zf.namelist() if any(s in n for s in VN_SPEAKERS)]
                    zf.extractall(target, members=members)
                    log(f"Giải nén {len(members)} file từ {os.path.basename(z)}")
            except Exception as e:
                log(f"Không giải nén được {z}: {e}")

    # Tự động tải từ Hugging Face mirror nếu chưa có đủ 4 speaker Việt Nam (hoàn toàn miễn phí, không cần form)
    import urllib.request
    base_hf_url = "https://huggingface.co/datasets/chikingsley/l2-arctic-release-v5.0/resolve/main"
    for spk in VN_SPEAKERS:
        has_tgs = glob.glob(os.path.join(target, "**", spk, "annotation", "*.TextGrid"), recursive=True)
        if not has_tgs:
            spk_zip = os.path.join(target, f"{spk}.zip")
            if not os.path.exists(spk_zip):
                url = f"{base_hf_url}/{spk}.zip"
                log(f"Đang tự động tải {spk}.zip (L2-ARCTIC Vietnamese) từ Hugging Face mirror: {url} ...")
                try:
                    urllib.request.urlretrieve(url, spk_zip)
                    log(f"Tải thành công {spk}.zip ({os.path.getsize(spk_zip)/1024/1024:.1f} MB)")
                except Exception as e:
                    log(f"Không thể tải tự động {spk}.zip từ HF: {e}")
            if os.path.exists(spk_zip):
                try:
                    with zipfile.ZipFile(spk_zip) as zf:
                        zf.extractall(target)
                    log(f"Giải nén thành công {spk}.zip vào {target}")
                except Exception as e:
                    log(f"Lỗi giải nén {spk_zip}: {e}")

    for inner in glob.glob(os.path.join(target, "**", "*.zip"), recursive=True):
        try:
            with zipfile.ZipFile(inner) as zf:
                zf.extractall(os.path.dirname(inner))
        except Exception as e:
            log(f"Không giải nén được {inner}: {e}")
    roots.append(target)
    return roots


def load_l2arctic_vn():
    import soundfile as sf
    items = []
    seen = set()
    for root in extract_l2arctic_archives():
        for spk in VN_SPEAKERS:
            for tg in glob.glob(os.path.join(root, "**", spk, "annotation", "*.TextGrid"), recursive=True):
                stem = os.path.splitext(os.path.basename(tg))[0]
                key = (spk, stem)
                if key in seen:
                    continue
                wav_path = os.path.join(os.path.dirname(os.path.dirname(tg)), "wav", stem + ".wav")
                if not os.path.exists(wav_path):
                    continue
                try:
                    seq = parse_l2arctic_phones(tg)
                    if not seq:
                        continue
                    wav, sr = sf.read(wav_path, dtype="float32", always_2d=False)
                    items.append({"wav": to_16k_mono(wav, sr), "seq": seq, "speaker": spk})
                    seen.add(key)
                except Exception as e:
                    log(f"Bỏ qua {tg}: {e}")
    if items:
        log(f"L2-ARCTIC tiếng Việt: đã nạp thành công {len(items)} câu phát âm có chú thích lỗi âm vị (HQTV, PNV, THV, TLV).")
    else:
        log("Không tìm thấy mẫu L2-ARCTIC -> Giai đoạn 2 tiếp tục với 100% SpeechOcean762.")
    return items


def collate_wavs(wavs):
    L = max(len(w) for w in wavs)
    x = np.zeros((len(wavs), L), np.float32)
    m = np.zeros((len(wavs), L), np.int64)
    for i, w in enumerate(wavs):
        w = (w - w.mean()) / (w.std() + 1e-7)
        x[i, : len(w)] = w
        m[i, : len(w)] = 1
    return torch.from_numpy(x), torch.from_numpy(m)


def make_batches(lengths, bs, shuffle, rng=None):
    idx = list(range(len(lengths)))
    if shuffle:
        rng.shuffle(idx)
    batches = []
    for c in [idx[i : i + bs * 50] for i in range(0, len(idx), bs * 50)]:
        c = sorted(c, key=lambda j: lengths[j])
        batches += [c[i : i + bs] for i in range(0, len(c), bs)]
    if shuffle:
        rng.shuffle(batches)
    return batches


try:
    from datasets import load_dataset, Audio
    t0 = time.time()
    so = load_dataset("mispeech/speechocean762", cache_dir=os.path.join(LOCAL_ROOT, "hf_cache"))
    so = so.cast_column("audio", Audio(decode=False))
    SO_TRAIN = prepare_split(so["train"], "train")
    SO_TEST = prepare_split(so["test"], "test")
    arr = np.array([it["scores"] for it in SO_TRAIN], dtype=np.float32)
    SCORE_MAX = [10.0 if float(arr[:, j].max()) > 1.5 else 1.0 for j in range(len(SCORE_FIELDS))]
    for it in SO_TRAIN + SO_TEST:
        it["y"] = np.clip(np.array(it["scores"], np.float32) / np.array(SCORE_MAX, np.float32), 0.0, 1.0)
    DATA_OK = len(SO_TRAIN) > 100 and len(SO_TEST) > 100
    log(f"Dữ liệu SpeechOcean762 sẵn sàng sau {time.time() - t0:.0f}s | SCORE_MAX={SCORE_MAX}")
    set_status("data_speechocean762", "ok" if DATA_OK else "too_few_samples")
    try:
        bk = os.path.join(DRIVE_ROOT, "datasets_backup", "speechocean762")
        if not os.path.exists(os.path.join(bk, "dataset_dict.json")):
            so.save_to_disk(bk)
            log(f"Đã sao lưu SpeechOcean762 vào {bk}")
    except Exception as e:
        log(f"Sao lưu dataset lên Drive thất bại (không ảnh hưởng huấn luyện): {e}")
except Exception:
    log(traceback.format_exc())
    set_status("data_speechocean762", "failed")

try:
    L2_ITEMS = load_l2arctic_vn()
    set_status("data_l2arctic_vn", f"{len(L2_ITEMS)} utterances")
except Exception:
    log(traceback.format_exc())
    set_status("data_l2arctic_vn", "failed")

## Ô 5 — GIAI ĐOẠN 1: Wav2Vec2 chấm điểm phát âm (5 chỉ số, xuất basis points 0..10000)

In [ ]:
import torch.nn as nn
from transformers import Wav2Vec2Model, get_cosine_schedule_with_warmup

P1_NAME = "phase1_scorer"


class PronunciationScorer(nn.Module):
    def __init__(self, base, n_out):
        super().__init__()
        self.backbone = Wav2Vec2Model.from_pretrained(base, layerdrop=0.0, mask_time_prob=0.05)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Linear(h, 512), nn.GELU(), nn.Dropout(0.1), nn.Linear(512, n_out))

    def forward(self, x, mask):
        hs = self.backbone(x, attention_mask=mask).last_hidden_state
        lengths = self.backbone._get_feat_extract_output_lengths(mask.sum(-1)).long()
        fm = (torch.arange(hs.shape[1], device=hs.device)[None, :] < lengths[:, None]).to(hs.dtype)
        pooled = (hs * fm.unsqueeze(-1)).sum(1) / fm.sum(1, keepdim=True).clamp(min=1)
        return torch.sigmoid(self.head(pooled.float()))


@torch.no_grad()
def evaluate_scorer(model):
    model.eval()
    preds, ys = [], []
    lens = [len(it["wav"]) for it in SO_TEST]
    for b in make_batches(lens, 16, False):
        x, m = collate_wavs([SO_TEST[i]["wav"] for i in b])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p = model(x.cuda(), m.cuda())
        preds.append(p.float().cpu().numpy())
        ys.append(np.stack([SO_TEST[i]["y"] for i in b]))
    P, Y = np.concatenate(preds), np.concatenate(ys)
    out = {}
    for j, f in enumerate(SCORE_FIELDS):
        c = np.corrcoef(P[:, j], Y[:, j])[0, 1]
        out[f] = float(np.nan_to_num(c))
    out["mae_total_bp"] = float(np.mean(np.abs(P[:, -1] - Y[:, -1])) * 10000)
    return out


def run_phase1():
    out_dir = os.path.join(LOCAL_ROOT, P1_NAME)
    os.makedirs(out_dir, exist_ok=True)
    rng = random.Random(SEED)
    torch.manual_seed(SEED)
    model = PronunciationScorer(BASE_W2V, len(SCORE_FIELDS)).cuda()
    model.backbone.freeze_feature_encoder()
    opt = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": P1_LR_BACKBONE},
        {"params": model.head.parameters(), "lr": P1_LR_HEAD},
    ], weight_decay=0.01)
    tr_len = [len(it["wav"]) for it in SO_TRAIN]
    total = math.ceil(len(SO_TRAIN) / P1_BATCH) * P1_EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, int(0.08 * total), total)
    best, history = -1.0, []
    for ep in range(1, P1_EPOCHS + 1):
        model.train()
        t0, losses = time.time(), []
        for b in make_batches(tr_len, P1_BATCH, True, rng):
            x, m = collate_wavs([SO_TRAIN[i]["wav"] for i in b])
            y = torch.from_numpy(np.stack([SO_TRAIN[i]["y"] for i in b])).cuda()
            with torch.autocast("cuda", dtype=torch.bfloat16):
                pred = model(x.cuda(non_blocking=True), m.cuda(non_blocking=True))
            loss = nn.functional.mse_loss(pred.float(), y)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())
        metrics = evaluate_scorer(model)
        mean_pcc = float(np.mean([metrics[f] for f in SCORE_FIELDS]))
        history.append({"epoch": ep, "train_mse": float(np.mean(losses)) if losses else None, **metrics})
        log(f"[GĐ1] Epoch {ep}/{P1_EPOCHS} | loss {history[-1]['train_mse']:.4f} | PCC total {metrics['total']:.3f} "
            f"| accuracy {metrics['accuracy']:.3f} | fluency {metrics['fluency']:.3f} | prosodic {metrics['prosodic']:.3f} "
            f"| MAE total {metrics['mae_total_bp']:.0f} bp | {time.time() - t0:.0f}s")
        if mean_pcc > best:
            best = mean_pcc
            model.backbone.save_pretrained(os.path.join(out_dir, "backbone"))
            torch.save(model.head.state_dict(), os.path.join(out_dir, "head.pt"))
            with open(os.path.join(out_dir, "scorer_config.json"), "w", encoding="utf-8") as f:
                json.dump({
                    "base_model": BASE_W2V, "score_fields": SCORE_FIELDS, "score_max_raw": SCORE_MAX,
                    "output": "sigmoid 0..1, nhân 10000 = basis points", "pooling": "masked mean",
                    "head": "Linear(h,512)-GELU-Dropout(0.1)-Linear(512,5)-Sigmoid",
                    "input": "16kHz mono, chuẩn hóa zero-mean unit-variance theo từng câu",
                    "best_epoch": ep, "test_metrics": metrics,
                }, f, indent=2, ensure_ascii=False)
            log(f"[GĐ1] Lưu checkpoint tốt nhất (mean PCC {best:.3f})")
    with open(os.path.join(out_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    del model, opt
    free_gpu()
    sync_to_drive(out_dir, P1_NAME)
    mark_done(P1_NAME, {"best_mean_pcc": best, "finished_at": datetime.now().isoformat()})
    return best


if not RUN_PHASE1:
    log("[GĐ1] Bỏ qua theo cấu hình.")
elif phase_done(P1_NAME):
    log("[GĐ1] Đã hoàn thành ở phiên trước -> bỏ qua.")
elif not DATA_OK:
    log("[GĐ1] Không có dữ liệu SpeechOcean762 -> bỏ qua.")
    set_status(P1_NAME, "skipped_no_data")
else:
    try:
        t0 = time.time()
        best = run_phase1()
        set_status(P1_NAME, f"ok (mean PCC {best:.3f}, {(time.time() - t0) / 60:.0f} phút)")
    except Exception:
        log(traceback.format_exc())
        set_status(P1_NAME, "failed")
    finally:
        free_gpu()

## Ô 6 — GIAI ĐOẠN 2: Wav2Vec2 nhận dạng âm vị thực tế (CTC) để phát hiện lỗi thay thế / nuốt âm

In [ ]:
from transformers import Wav2Vec2ForCTC, get_linear_schedule_with_warmup

P2_NAME = "phase2_phoneme_ctc"
ARPABET_TO_IPA = {
    "AA": "ɑː", "AE": "æ", "AH": "ʌ", "AO": "ɔː", "AW": "aʊ", "AY": "aɪ", "B": "b", "CH": "tʃ", "D": "d",
    "DH": "ð", "EH": "e", "ER": "ɜː", "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "iː",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "N": "n", "NG": "ŋ", "OW": "əʊ", "OY": "ɔɪ", "P": "p",
    "R": "r", "S": "s", "SH": "ʃ", "T": "t", "TH": "θ", "UH": "ʊ", "UW": "uː", "V": "v", "W": "w",
    "Y": "j", "Z": "z", "ZH": "ʒ", "SPN": "?",
}


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]


def ctc_greedy(ids):
    out, prev = [], None
    for i in ids:
        if i != prev and i != 0:
            out.append(i)
        prev = i
    return out


@torch.no_grad()
def evaluate_per(model, data):
    model.eval()
    errs, total = 0, 0
    lens = [len(w) for w, _ in data]
    for b in make_batches(lens, 16, False):
        x, m = collate_wavs([data[i][0] for i in b])
        x, m = x.cuda(), m.cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x, attention_mask=m).logits
        flens = model._get_feat_extract_output_lengths(m.sum(-1)).long().tolist()
        ids = logits.argmax(-1).cpu().tolist()
        for k, i in enumerate(b):
            hyp = ctc_greedy(ids[k][: flens[k]])
            ref = data[i][1]
            errs += edit_distance(ref, hyp)
            total += len(ref)
    return errs / max(total, 1)


def run_phase2():
    out_dir = os.path.join(LOCAL_ROOT, P2_NAME)
    os.makedirs(out_dir, exist_ok=True)
    rng = random.Random(SEED)
    torch.manual_seed(SEED)
    vn_train, vn_test = [], []
    by_spk = {}
    for it in L2_ITEMS:
        by_spk.setdefault(it["speaker"], []).append(it)
    for spk, items in by_spk.items():
        rng.shuffle(items)
        vn_test += items[:20]
        vn_train += items[20:]
    all_seqs = [it["seq"] for it in SO_TRAIN + SO_TEST + L2_ITEMS]
    phones = sorted({p for s in all_seqs for p in s})
    vocab = {"[PAD]": 0, "[UNK]": 1}
    for p in phones:
        vocab[p] = len(vocab)
    enc = lambda seq: [vocab.get(p, 1) for p in seq]
    train = [(it["wav"], enc(it["seq"])) for it in SO_TRAIN + vn_train if it["seq"]]
    test = [(it["wav"], enc(it["seq"])) for it in SO_TEST if it["seq"]]
    test_vn = [(it["wav"], enc(it["seq"])) for it in vn_test if it["seq"]]
    log(f"[GĐ2] vocab {len(vocab)} | train {len(train)} (VN {len(vn_train)}) | test {len(test)} | test VN {len(test_vn)}")
    with open(os.path.join(out_dir, "vocab.json"), "w", encoding="utf-8") as f:
        json.dump(vocab, f, indent=2, ensure_ascii=False)
    with open(os.path.join(out_dir, "arpabet_to_ipa.json"), "w", encoding="utf-8") as f:
        json.dump(ARPABET_TO_IPA, f, indent=2, ensure_ascii=False)

    model = Wav2Vec2ForCTC.from_pretrained(
        BASE_W2V, vocab_size=len(vocab), pad_token_id=0, ctc_loss_reduction="mean", ctc_zero_infinity=True,
        layerdrop=0.0, mask_time_prob=0.05, ignore_mismatched_sizes=True,
    ).cuda()
    model.freeze_feature_encoder()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=P2_LR, weight_decay=0.0)
    total = math.ceil(len(train) / P2_BATCH) * P2_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, min(500, total // 10), total)
    lens = [len(w) for w, _ in train]
    best, history = 9.9, []
    for ep in range(1, P2_EPOCHS + 1):
        model.train()
        t0, losses = time.time(), []
        for b in make_batches(lens, P2_BATCH, True, rng):
            x, m = collate_wavs([train[i][0] for i in b])
            L = max(len(train[i][1]) for i in b)
            labels = torch.full((len(b), L), -100, dtype=torch.long)
            for k, i in enumerate(b):
                labels[k, : len(train[i][1])] = torch.tensor(train[i][1])
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = model(x.cuda(non_blocking=True), attention_mask=m.cuda(non_blocking=True), labels=labels.cuda()).loss
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            losses.append(loss.item())
        msg = f"[GĐ2] Epoch {ep}/{P2_EPOCHS} | CTC loss {np.mean(losses):.4f} | {time.time() - t0:.0f}s"
        if ep % P2_EVAL_EVERY == 0 or ep == P2_EPOCHS:
            per = evaluate_per(model, test)
            per_vn = evaluate_per(model, test_vn) if test_vn else None
            history.append({"epoch": ep, "loss": float(np.mean(losses)), "per": per, "per_vn": per_vn})
            msg += f" | PER test {per * 100:.1f}%" + (f" | PER VN {per_vn * 100:.1f}%" if per_vn is not None else "")
            if per < best:
                best = per
                model.save_pretrained(os.path.join(out_dir, "model"))
                with open(os.path.join(out_dir, "phoneme_config.json"), "w", encoding="utf-8") as f:
                    json.dump({"base_model": BASE_W2V, "blank_id": 0, "best_epoch": ep, "per_test": per,
                               "per_vn": per_vn, "decode": "greedy CTC, bỏ lặp và blank(0)",
                               "input": "16kHz mono, chuẩn hóa zero-mean unit-variance theo từng câu"},
                              f, indent=2, ensure_ascii=False)
                msg += " | đã lưu checkpoint tốt nhất"
        log(msg)
    with open(os.path.join(out_dir, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    del model, opt
    free_gpu()
    sync_to_drive(out_dir, P2_NAME)
    mark_done(P2_NAME, {"best_per": best, "finished_at": datetime.now().isoformat()})
    return best


if not RUN_PHASE2:
    log("[GĐ2] Bỏ qua theo cấu hình.")
elif phase_done(P2_NAME):
    log("[GĐ2] Đã hoàn thành ở phiên trước -> bỏ qua.")
elif not DATA_OK:
    log("[GĐ2] Không có dữ liệu -> bỏ qua.")
    set_status(P2_NAME, "skipped_no_data")
else:
    try:
        t0 = time.time()
        best = run_phase2()
        set_status(P2_NAME, f"ok (PER {best * 100:.1f}%, {(time.time() - t0) / 60:.0f} phút)")
    except Exception:
        log(traceback.format_exc())
        set_status(P2_NAME, "failed")
    finally:
        free_gpu()

## Ô 7 — Ghi script Giai đoạn 3 (chạy trong tiến trình riêng để cô lập lỗi)

In [ ]:
%%writefile /content/phase3_llama.py
import os, sys, json, random, inspect, traceback, shutil, glob, time
from datetime import datetime

OUT, DRIVE_OUT = sys.argv[1], sys.argv[2]
MAX_STEPS, N_SAMPLES = int(sys.argv[3]), int(sys.argv[4])
EXPORT_GGUF, LOG_FILE = sys.argv[5] == "1", sys.argv[6]
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)


def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [GĐ3] {msg}"
    print(line, flush=True)
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass


USE_UNSLOTH = True
try:
    import unsloth  # noqa: F401  (phải import trước transformers)
    from unsloth import FastLanguageModel
except Exception as e:
    USE_UNSLOTH = False
    log(f"Không dùng được Unsloth ({e}) -> chuyển sang chế độ dự phòng transformers + peft (bf16 LoRA).")

import torch
from datasets import Dataset

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SYSTEM_PROMPT = (
    "Bạn là Cố Vấn Ngữ Âm AI cho người Việt học tiếng Anh (chuẩn British RP). "
    "Nhận dữ liệu phân tích âm vị dạng JSON và CHỈ trả về một JSON hợp lệ gồm: "
    "summaryVi (string), tipsVi (mảng string), visemeTimeline (mảng {ipa, startMs, transitionMs, holdMs, "
    "morphWeights (số nguyên 0..1000), highlightParts}). Giải thích vị trí môi, răng, lưỡi, luồng hơi bằng tiếng Việt tự nhiên."
)

PROFILES = [
    dict(target="θ", actual="t", etype="substitution",
         words=["think", "three", "thank", "bath", "mouth", "nothing", "birthday", "teeth", "month", "thick"],
         cause="tiếng Việt không có âm răng xát /θ/ nên người học thường thay bằng /t/ hoặc 'th' tiếng Việt",
         tips=["Đặt đầu lưỡi nhẹ giữa hai hàm răng cửa, lộ ra khoảng 2–3 mm.",
               "Thổi luồng hơi liên tục qua khe giữa lưỡi và răng trên, không bật hơi như /t/.",
               "Giữ dây thanh không rung: đặt tay lên cổ họng để kiểm tra.",
               "Luyện cặp tối thiểu: tin – thin, tree – three, tank – thank."],
         morph={"jawOpen": 220, "tongueInterdental": 950, "lipSpread": 300},
         parts=["tongue_tip", "upper_teeth", "lower_teeth"]),
    dict(target="ð", actual="d", etype="substitution",
         words=["this", "that", "the", "mother", "weather", "breathe", "those", "father", "with", "other"],
         cause="âm /ð/ hữu thanh giữa hai răng thường bị thay bằng /d/ tiếng Việt",
         tips=["Đặt đầu lưỡi giữa hai hàm răng như âm /θ/.",
               "Cho dây thanh rung trong khi hơi thoát nhẹ qua kẽ răng.",
               "Không chạm lưỡi vào nướu răng trên như khi nói /d/.",
               "Luyện cặp: day – they, dare – there."],
         morph={"jawOpen": 200, "tongueInterdental": 900, "voicing": 900},
         parts=["tongue_tip", "upper_teeth", "larynx"]),
    dict(target="ð", actual="z", etype="substitution",
         words=["this", "they", "there", "smooth", "clothing", "brother", "either", "together"],
         cause="lưỡi bị rụt vào sau răng nên âm /ð/ biến thành /z/",
         tips=["Đưa đầu lưỡi ra trước, chạm nhẹ mép răng cửa trên.",
               "Giữ rung dây thanh nhưng không để hơi rít qua khe răng như /z/.",
               "Nói chậm 'the – the – the' trước gương để quan sát lưỡi."],
         morph={"jawOpen": 200, "tongueInterdental": 880, "voicing": 900},
         parts=["tongue_tip", "upper_teeth", "larynx"]),
    dict(target="t", actual="", etype="omission",
         words=["cat", "hot", "what", "street", "night", "eight", "light", "fruit", "boat", "start"],
         cause="âm cuối /t/ trong tiếng Việt không bật hơi nên dễ bị nuốt mất",
         tips=["Kết thúc từ bằng cách chạm đầu lưỡi vào nướu răng trên.",
               "Bật nhẹ một luồng hơi ngắn sau khi chạm lưỡi.",
               "Luyện nối âm: 'what is' đọc thành 'wha-tis'.",
               "Đặt tờ giấy trước miệng: giấy phải rung nhẹ ở cuối từ."],
         morph={"jawOpen": 150, "tongueAlveolar": 950},
         parts=["tongue_tip", "alveolar_ridge"]),
    dict(target="k", actual="", etype="omission",
         words=["book", "like", "back", "desk", "black", "week", "milk", "cake", "walk", "speak"],
         cause="phụ âm cuối /k/ bị nuốt do thói quen dừng âm của tiếng Việt",
         tips=["Nâng cuống lưỡi chạm ngạc mềm ở cuối từ.",
               "Bật hơi dứt khoát sau khi cuống lưỡi rời ngạc mềm.",
               "Luyện chậm: boo – k, boo-k, book."],
         morph={"jawOpen": 300, "palateContact": 900},
         parts=["tongue_back", "velum"]),
    dict(target="s", actual="", etype="omission",
         words=["cats", "books", "bus", "likes", "works", "class", "nice", "yes", "rice", "house"],
         cause="âm /s/ cuối từ (số nhiều, ngôi thứ ba) thường bị bỏ qua",
         tips=["Khép hai hàm răng gần nhau, đầu lưỡi gần nướu răng trên.",
               "Đẩy luồng hơi rít liên tục ra phía trước qua kẽ răng.",
               "Kéo dài âm /s/ thêm nửa nhịp để nghe rõ đuôi từ."],
         morph={"jawOpen": 120, "tongueAlveolar": 700, "lipSpread": 500},
         parts=["tongue_tip", "alveolar_ridge", "upper_teeth"]),
    dict(target="z", actual="s", etype="substitution",
         words=["zoo", "zero", "is", "has", "roses", "music", "easy", "busy", "lazy", "prize"],
         cause="âm /z/ bị mất rung dây thanh nên nghe thành /s/",
         tips=["Giữ vị trí lưỡi như /s/ nhưng cho dây thanh rung.",
               "Đặt tay lên cổ: phải cảm nhận được rung khi nói /z/.",
               "Luyện cặp: sip – zip, bus – buzz."],
         morph={"jawOpen": 120, "tongueAlveolar": 700, "voicing": 900},
         parts=["tongue_tip", "alveolar_ridge", "larynx"]),
    dict(target="ʃ", actual="s", etype="substitution",
         words=["she", "shop", "fish", "wash", "sure", "nation", "shoe", "push", "ship", "special"],
         cause="âm /ʃ/ bị phát âm thành /s/ do lưỡi đặt quá gần răng",
         tips=["Chu tròn môi ra phía trước.",
               "Lùi mặt lưỡi về phía ngạc cứng, rộng hơn khi nói /s/.",
               "Luyện cặp: sip – ship, see – she."],
         morph={"lipRound": 700, "tongueBlade": 850, "jawOpen": 180},
         parts=["lips", "tongue_blade", "hard_palate"]),
    dict(target="ʒ", actual="z", etype="substitution",
         words=["measure", "vision", "usual", "television", "garage", "pleasure", "decision"],
         cause="âm /ʒ/ hiếm gặp nên thường bị thay bằng /z/",
         tips=["Chu môi và đặt lưỡi như âm /ʃ/.",
               "Thêm rung dây thanh để thành /ʒ/.",
               "Luyện chậm: mea – ʒure, vi – ʒion."],
         morph={"lipRound": 700, "tongueBlade": 850, "voicing": 900},
         parts=["lips", "tongue_blade", "larynx"]),
    dict(target="dʒ", actual="z", etype="substitution",
         words=["job", "judge", "orange", "large", "June", "jacket", "bridge", "page", "village", "giant"],
         cause="âm tắc xát /dʒ/ bị thay bằng âm 'gi/d' tiếng Việt (gần /z/)",
         tips=["Bắt đầu bằng việc chạm mặt lưỡi vào ngạc cứng như /d/.",
               "Nhả lưỡi từ từ để hơi xát ra như /ʒ/, môi hơi chu.",
               "Luyện cặp: zoo – Jew, zip – gyp."],
         morph={"lipRound": 500, "tongueBlade": 900, "palateContact": 800},
         parts=["tongue_blade", "hard_palate", "lips"]),
    dict(target="tʃ", actual="ʃ", etype="substitution",
         words=["chair", "teacher", "watch", "church", "lunch", "cheese", "much", "kitchen"],
         cause="thiếu pha tắc ở đầu âm /tʃ/ nên nghe thành /ʃ/",
         tips=["Chặn hơi hoàn toàn bằng mặt lưỡi ở ngạc cứng trước.",
               "Bật ra đột ngột thành tiếng xát /ʃ/.",
               "Luyện cặp: share – chair, wash – watch."],
         morph={"lipRound": 500, "tongueBlade": 900, "palateContact": 850},
         parts=["tongue_blade", "hard_palate", "lips"]),
    dict(target="n", actual="l", etype="substitution",
         words=["night", "no", "need", "name", "know", "nine", "nose", "never"],
         cause="nhầm lẫn /n/ – /l/ thường gặp ở người học miền Bắc",
         tips=["Hạ ngạc mềm để hơi thoát ra đường mũi.",
               "Bịt mũi thử: nếu âm bị tắc nghĩa là đang nói đúng /n/.",
               "Luyện cặp: light – night, low – no."],
         morph={"velumLower": 900, "tongueAlveolar": 950, "jawOpen": 200},
         parts=["velum", "tongue_tip", "nasal_cavity"]),
    dict(target="r", actual="z", etype="substitution",
         words=["red", "right", "very", "sorry", "room", "rice", "read", "road"],
         cause="âm /r/ tiếng Anh bị thay bằng 'r/d/gi' (gần /z/) theo phương ngữ miền Bắc",
         tips=["Cong nhẹ đầu lưỡi lên nhưng KHÔNG chạm vòm miệng.",
               "Hơi chu môi khi bắt đầu âm /r/.",
               "Không để hơi rít qua kẽ răng như /z/."],
         morph={"tongueRetroflex": 800, "lipRound": 450, "jawOpen": 250},
         parts=["tongue_tip", "lips"]),
    dict(target="v", actual="j", etype="substitution",
         words=["very", "van", "voice", "love", "five", "river", "visit", "video"],
         cause="âm /v/ bị đọc thành /j/ theo phương ngữ miền Nam",
         tips=["Đặt răng cửa trên chạm nhẹ môi dưới.",
               "Cho hơi xát qua khe môi – răng và rung dây thanh.",
               "Luyện cặp: yet – vet, yes – vest."],
         morph={"lowerLipToTeeth": 950, "voicing": 900, "jawOpen": 150},
         parts=["lower_lip", "upper_teeth"]),
    dict(target="iː", actual="ɪ", etype="substitution",
         words=["sheep", "seat", "feel", "leave", "green", "beach", "team", "need"],
         cause="nguyên âm dài /iː/ bị đọc ngắn thành /ɪ/",
         tips=["Kéo căng môi sang hai bên như đang cười.",
               "Giữ âm dài gấp đôi so với /ɪ/.",
               "Luyện cặp: ship – sheep, sit – seat."],
         morph={"lipSpread": 850, "jawOpen": 150, "tongueHigh": 900},
         parts=["lips", "tongue_body"]),
    dict(target="æ", actual="e", etype="substitution",
         words=["cat", "bad", "man", "apple", "black", "hand", "happy", "bag"],
         cause="âm /æ/ bị đọc thành 'e' tiếng Việt do miệng mở chưa đủ rộng",
         tips=["Mở miệng rộng hơn, hạ hàm dưới xuống.",
               "Đặt lưỡi thấp, đầu lưỡi chạm răng cửa dưới.",
               "Luyện cặp: bed – bad, men – man."],
         morph={"jawOpen": 800, "tongueLow": 850, "lipSpread": 400},
         parts=["jaw", "tongue_body"]),
    dict(target="d", actual="", etype="omission",
         words=["good", "bed", "red", "played", "friend", "cold", "find", "food"],
         cause="âm cuối /d/ bị nuốt hoặc đọc thành /t/ không bật",
         tips=["Chạm đầu lưỡi vào nướu răng trên ở cuối từ.",
               "Giữ rung dây thanh khi kết thúc âm /d/.",
               "Luyện đuôi -ed: played = play + /d/."],
         morph={"jawOpen": 150, "tongueAlveolar": 950, "voicing": 850},
         parts=["tongue_tip", "alveolar_ridge", "larynx"]),
]

SUMMARY_SUB = [
    'Âm /{t}/ trong từ "{w}" đang bị phát âm thành /{a}/ (độ chính xác {pct}%): {cause}.',
    'Bạn đọc "{w}" với âm /{a}/ thay cho /{t}/ ({pct}%). Nguyên nhân: {cause}.',
    'Lỗi thay thế âm: /{t}/ → /{a}/ trong "{w}" ({pct}%), vì {cause}.',
]
SUMMARY_OMI = [
    'Âm cuối /{t}/ trong từ "{w}" bị nuốt mất (độ chính xác {pct}%): {cause}.',
    'Bạn chưa phát âm âm /{t}/ ở cuối "{w}" ({pct}%). Nguyên nhân: {cause}.',
    'Lỗi bỏ âm: thiếu /{t}/ trong "{w}" ({pct}%), vì {cause}.',
]
SUMMARY_OK = [
    'Âm /{t}/ trong từ "{w}" được phát âm rất tốt ({pct}%). Hãy giữ khẩu hình này.',
    'Tuyệt vời! "{w}" có âm /{t}/ chuẩn xác ({pct}%).',
]


def make_sample(rng):
    p = rng.choice(PROFILES)
    w = rng.choice(p["words"])
    correct = rng.random() < 0.12
    gop = rng.randint(8200, 9800) if correct else rng.randint(2500, 6900)
    start = rng.randrange(80, 900, 10)
    inp = {"word": w, "targetIpa": p["target"], "actualIpa": p["target"] if correct else p["actual"],
           "errorType": "correct" if correct else p["etype"], "gopScoreBp": gop, "startMs": start}
    pct = round(gop / 100)
    if correct:
        summary = rng.choice(SUMMARY_OK).format(t=p["target"], w=w, pct=pct)
        tips = ["Duy trì vị trí cơ quan phát âm hiện tại và luyện thêm ở tốc độ nói tự nhiên."] + rng.sample(p["tips"], 1)
    else:
        pool = SUMMARY_OMI if p["etype"] == "omission" else SUMMARY_SUB
        summary = rng.choice(pool).format(t=p["target"], a=p["actual"], w=w, pct=pct, cause=p["cause"])
        tips = rng.sample(p["tips"], k=min(len(p["tips"]), rng.choice([2, 3])))
    morph = {k: max(0, min(1000, v + rng.randint(-40, 40))) for k, v in p["morph"].items()}
    out = {"summaryVi": summary, "tipsVi": tips, "visemeTimeline": [{
        "ipa": p["target"], "startMs": start, "transitionMs": rng.choice([30, 40, 50, 60]),
        "holdMs": rng.choice([60, 80, 100, 120, 140]), "morphWeights": morph, "highlightParts": p["parts"]}]}
    user = "Phân tích lỗi phát âm sau và trả về JSON phản hồi:\n" + json.dumps(inp, ensure_ascii=False)
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": user},
                         {"role": "assistant", "content": json.dumps(out, ensure_ascii=False)}]}


def build_dataset(n, seed=42):
    rng = random.Random(seed)
    return [make_sample(rng) for _ in range(n)]


def sync(name):
    src = os.path.join(OUT, name)
    if os.path.exists(src):
        shutil.copytree(src, os.path.join(DRIVE_OUT, name), dirs_exist_ok=True)
        log(f"Đã chép {name} lên Drive")


def quick_test(model, tokenizer):
    try:
        probe = {"word": "think", "targetIpa": "θ", "actualIpa": "t", "errorType": "substitution",
                 "gopScoreBp": 5400, "startMs": 200}
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": "Phân tích lỗi phát âm sau và trả về JSON phản hồi:\n" + json.dumps(probe, ensure_ascii=False)}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
        if not torch.is_tensor(ids):
            ids = ids["input_ids"]
        ids = ids.to("cuda")
        with torch.no_grad():
            out = model.generate(input_ids=ids, max_new_tokens=400, do_sample=False)
        txt = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
        try:
            json.loads(txt)
            valid = True
        except Exception:
            valid = False
        log(f"Kiểm tra sinh thử (JSON hợp lệ = {valid}):\n{txt[:800]}")
        with open(os.path.join(OUT, "quick_test.json"), "w", encoding="utf-8") as f:
            json.dump({"input": probe, "output": txt, "json_valid": valid}, f, indent=2, ensure_ascii=False)
        shutil.copy(os.path.join(OUT, "quick_test.json"), os.path.join(DRIVE_OUT, "quick_test.json"))
    except Exception:
        log("Sinh thử thất bại (không ảnh hưởng trọng số đã lưu):\n" + traceback.format_exc())


def write_modelfile(gguf_dir, gguf_name):
    with open(os.path.join(gguf_dir, "Modelfile"), "w", encoding="utf-8") as f:
        f.write(f"FROM ./{gguf_name}\n")
        f.write("PARAMETER temperature 0.3\n")
        f.write("PARAMETER num_ctx 2048\n")
        f.write('SYSTEM """' + SYSTEM_PROMPT + '"""\n')


def train_unsloth(texts):
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None
    from transformers import TrainingArguments

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", max_seq_length=1024, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=TARGET_MODULES, lora_alpha=16, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42)
    ds = Dataset.from_dict({"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in texts]})

    kw = dict(per_device_train_batch_size=8, gradient_accumulation_steps=2, warmup_steps=20, max_steps=MAX_STEPS,
              learning_rate=2e-4, logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
              lr_scheduler_type="linear", seed=42, output_dir=os.path.join(OUT, "trainer_out"),
              report_to="none", bf16=True, save_strategy="no")
    if SFTConfig is not None:
        params = inspect.signature(SFTConfig.__init__).parameters
        if "dataset_text_field" in params:
            kw["dataset_text_field"] = "text"
        if "max_seq_length" in params:
            kw["max_seq_length"] = 1024
        elif "max_length" in params:
            kw["max_length"] = 1024
        if "packing" in params:
            kw["packing"] = False
        args = SFTConfig(**kw)
    else:
        args = TrainingArguments(**kw)
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    tk = dict(model=model, train_dataset=ds, args=args)
    if "processing_class" in tparams:
        tk["processing_class"] = tokenizer
    else:
        tk["tokenizer"] = tokenizer
    if SFTConfig is None:
        tk["dataset_text_field"] = "text"
        tk["max_seq_length"] = 1024
    trainer = SFTTrainer(**tk)
    t0 = time.time()
    stats = trainer.train()
    log(f"Huấn luyện xong sau {(time.time() - t0) / 60:.0f} phút | {stats}")

    model.save_pretrained(os.path.join(OUT, "lora_adapter"))
    tokenizer.save_pretrained(os.path.join(OUT, "lora_adapter"))
    sync("lora_adapter")
    FastLanguageModel.for_inference(model)
    quick_test(model, tokenizer)

    exported = False
    if EXPORT_GGUF:
        try:
            log("Đang xuất GGUF q4_k_m (biên dịch llama.cpp, có thể mất 10–20 phút)...")
            model.save_pretrained_gguf(os.path.join(OUT, "gguf_raw"), tokenizer, quantization_method="q4_k_m")
            files = [f for f in glob.glob(os.path.join(OUT, "**", "*.gguf"), recursive=True)]
            files += [f for f in glob.glob(os.path.join(os.getcwd(), "*.gguf"))]
            q4 = [f for f in files if "q4_k_m" in os.path.basename(f).lower()] or files
            if q4:
                final_dir = os.path.join(OUT, "gguf")
                os.makedirs(final_dir, exist_ok=True)
                name = "llama3.1_8b_tutor_3d_q4_k_m.gguf"
                shutil.copy(q4[0], os.path.join(final_dir, name))
                write_modelfile(final_dir, name)
                sync("gguf")
                exported = True
            else:
                log("Không tìm thấy file .gguf sau khi xuất.")
        except Exception:
            log("Xuất GGUF thất bại:\n" + traceback.format_exc())
    if not exported:
        try:
            log("Lưu bản merged 16-bit dự phòng (~16GB)...")
            model.save_pretrained_merged(os.path.join(OUT, "merged_16bit"), tokenizer, save_method="merged_16bit")
            sync("merged_16bit")
        except Exception:
            log("Lưu merged 16-bit thất bại:\n" + traceback.format_exc())
    return {"backend": "unsloth", "gguf": exported}


def train_fallback(texts):
    from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
    from peft import LoraConfig, get_peft_model
    name = "unsloth/Meta-Llama-3.1-8B-Instruct"
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map={"": 0})
    model.config.use_cache = False
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0,
                                             target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
    enc = [tok(tok.apply_chat_template(m, tokenize=False), add_special_tokens=False,
               truncation=True, max_length=1024)["input_ids"] for m in texts]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=2e-4, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, 20, MAX_STEPS)
    rng = random.Random(42)
    model.train()
    t0 = time.time()
    for step in range(1, MAX_STEPS + 1):
        batch = rng.sample(enc, 8)
        L = max(len(b) for b in batch)
        ids = torch.full((8, L), tok.pad_token_id, dtype=torch.long)
        labels = torch.full((8, L), -100, dtype=torch.long)
        att = torch.zeros((8, L), dtype=torch.long)
        for i, b in enumerate(batch):
            ids[i, : len(b)] = torch.tensor(b)
            labels[i, : len(b)] = torch.tensor(b)
            att[i, : len(b)] = 1
        loss = model(input_ids=ids.cuda(), attention_mask=att.cuda(), labels=labels.cuda()).loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        sched.step()
        if step % 10 == 0:
            log(f"step {step}/{MAX_STEPS} | loss {loss.item():.4f} | {(time.time() - t0) / 60:.1f} phút")
    model.save_pretrained(os.path.join(OUT, "lora_adapter"))
    tok.save_pretrained(os.path.join(OUT, "lora_adapter"))
    sync("lora_adapter")
    model.eval()
    model.config.use_cache = True
    quick_test(model, tok)
    try:
        merged = model.merge_and_unload()
        merged.save_pretrained(os.path.join(OUT, "merged_16bit"), safe_serialization=True)
        tok.save_pretrained(os.path.join(OUT, "merged_16bit"))
        sync("merged_16bit")
    except Exception:
        log("Lưu merged 16-bit thất bại:\n" + traceback.format_exc())
    return {"backend": "transformers+peft", "gguf": False}


def main():
    samples = build_dataset(N_SAMPLES)
    ds_dir = os.path.join(OUT, "dataset")
    os.makedirs(ds_dir, exist_ok=True)
    with open(os.path.join(ds_dir, "tutor_sft.jsonl"), "w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    sync("dataset")
    log(f"Đã sinh {len(samples)} mẫu huấn luyện sư phạm. Backend: {'Unsloth' if USE_UNSLOTH else 'transformers+peft'}")
    msgs = [s["messages"] for s in samples]
    info = train_unsloth(msgs) if USE_UNSLOTH else train_fallback(msgs)
    info["finished_at"] = datetime.now().isoformat()
    with open(os.path.join(DRIVE_OUT, "_DONE.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    log(f"HOÀN TẤT: {info}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("LỖI:\n" + traceback.format_exc())
        sys.exit(1)

## Ô 8 — GIAI ĐOẠN 3: Cài Unsloth (có giới hạn thời gian) và chạy huấn luyện Llama 3.1 8B

In [ ]:
P3_NAME = "phase3_llama_tutor"

if not RUN_PHASE3:
    log("[GĐ3] Bỏ qua theo cấu hình.")
elif phase_done(P3_NAME):
    log("[GĐ3] Đã hoàn thành ở phiên trước -> bỏ qua.")
else:
    try:
        free_gpu()
        if importlib.util.find_spec("unsloth") is None:
            try:
                pip_install(["unsloth"], 1800)
            except Exception as e:
                log(f"[GĐ3] Cài Unsloth thất bại ({e}) -> script sẽ dùng chế độ dự phòng transformers + peft.")
        env = dict(os.environ, PYTHONUNBUFFERED="1")
        cmd = [sys.executable, "/content/phase3_llama.py", os.path.join(LOCAL_ROOT, P3_NAME),
               os.path.join(DRIVE_ROOT, P3_NAME), str(P3_MAX_STEPS), str(P3_NUM_SAMPLES),
               "1" if EXPORT_GGUF else "0", LOG_FILE]
        t0 = time.time()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in proc.stdout:
            print(line, end="", flush=True)
        rc = proc.wait()
        status = "ok" if rc == 0 and phase_done(P3_NAME) else f"failed (exit code {rc})"
        set_status(P3_NAME, f"{status}, {(time.time() - t0) / 60:.0f} phút")
        log(f"[GĐ3] Kết thúc: {status}")
    except Exception:
        log(traceback.format_exc())
        set_status(P3_NAME, "failed")

## Ô 9 — Tổng kết, ghi manifest + script nạp mô hình local, tự ngắt runtime

In [ ]:
LOCAL_LOADER = r"""
# load_models_local.py - nạp mô hình đã huấn luyện trên máy cá nhân (CPU hoặc GPU)
# pip install torch transformers soundfile librosa numpy
import json
import numpy as np
import torch
import torch.nn as nn
import soundfile as sf
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2ForCTC


def load_wav(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    return ((wav - wav.mean()) / (wav.std() + 1e-7)).astype(np.float32)


class Scorer(nn.Module):
    def __init__(self, d):
        super().__init__()
        with open(f"{d}/scorer_config.json", encoding="utf-8") as f:
            self.cfg = json.load(f)
        self.backbone = Wav2Vec2Model.from_pretrained(f"{d}/backbone")
        h, n = self.backbone.config.hidden_size, len(self.cfg["score_fields"])
        self.head = nn.Sequential(nn.Linear(h, 512), nn.GELU(), nn.Dropout(0.1), nn.Linear(512, n))
        self.head.load_state_dict(torch.load(f"{d}/head.pt", map_location="cpu"))
        self.eval()

    @torch.no_grad()
    def score(self, wav):
        hs = self.backbone(torch.from_numpy(wav)[None]).last_hidden_state.mean(1)
        p = torch.sigmoid(self.head(hs))[0].tolist()
        return {f: int(round(v * 10000)) for f, v in zip(self.cfg["score_fields"], p)}


class PhonemeRecognizer:
    def __init__(self, d):
        self.model = Wav2Vec2ForCTC.from_pretrained(f"{d}/model").eval()
        with open(f"{d}/vocab.json", encoding="utf-8") as f:
            self.inv = {v: k for k, v in json.load(f).items()}
        with open(f"{d}/arpabet_to_ipa.json", encoding="utf-8") as f:
            self.ipa = json.load(f)

    @torch.no_grad()
    def recognize(self, wav):
        ids = self.model(torch.from_numpy(wav)[None]).logits.argmax(-1)[0].tolist()
        out, prev = [], None
        for i in ids:
            if i != prev and i != 0:
                out.append(self.inv[i])
            prev = i
        return [{"arpabet": p, "ipa": self.ipa.get(p, p)} for p in out]


if __name__ == "__main__":
    wav = load_wav("sample.wav")
    print(Scorer("phase1_scorer").score(wav))
    print(PhonemeRecognizer("phase2_phoneme_ctc").recognize(wav))
"""

try:
    with open(os.path.join(DRIVE_ROOT, "load_models_local.py"), "w", encoding="utf-8") as f:
        f.write(LOCAL_LOADER.lstrip())
    status = load_status()
    manifest = {
        "system": "AI Speaking Tutor 3D",
        "hardware": f"{GPU_NAME} ({GPU_VRAM_GB:.0f} GB)",
        "finished_at": datetime.now().isoformat(),
        "status": status,
        "deliverables": {
            "phase1_scorer": "phase1_scorer/backbone + head.pt + scorer_config.json",
            "phase2_phoneme_ctc": "phase2_phoneme_ctc/model + vocab.json + arpabet_to_ipa.json",
            "phase3_llama_tutor": "phase3_llama_tutor/lora_adapter, gguf/llama3.1_8b_tutor_3d_q4_k_m.gguf (+ Modelfile) hoặc merged_16bit",
            "tts": "edge-tts en-GB-SoniaNeural / en-GB-RyanNeural (không cần huấn luyện)",
            "local_loader": "load_models_local.py",
        },
    }
    with open(os.path.join(DRIVE_ROOT, "metadata_manifest.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    log("=" * 70)
    log("TỔNG KẾT:")
    for k, v in status.items():
        log(f"  {k}: {v}")
    total_bytes = 0
    for root, _, files in os.walk(DRIVE_ROOT):
        for fn in files:
            total_bytes += os.path.getsize(os.path.join(root, fn))
    log(f"Tổng dung lượng trên Drive: {total_bytes / 1024 ** 3:.2f} GB tại {DRIVE_ROOT}")
except Exception:
    log(traceback.format_exc())

try:
    drive.flush_and_unmount()
    print("Đã đồng bộ và ngắt Google Drive an toàn.")
except Exception as e:
    print(f"flush_and_unmount: {e}")

if AUTO_DISCONNECT_WHEN_DONE:
    print("Tự ngắt runtime để tiết kiệm đơn vị điện toán...")
    from google.colab import runtime
    runtime.unassign()